In [1]:
from matplotlib import pyplot as plt
import json
import serial
import numpy as np
from datetime import datetime
import os
import time
from pytz import timezone
from PIL import Image, ImageSequence
import skimage
from typing import List
import serial, struct, time, collections
import numpy as np
import tkinter as tk
import time
import tkinter as tk
from tkinter import *
from tkinter import ttk,scrolledtext
import os
from matplotlib import pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
import numpy as np
import cv2
import threading
import nidaqmx
import nidaqmx.system
import queue
import math
from PIL import ImageTk, Image
import shutil
import tifffile
import matplotlib.pyplot as plt
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg
from tkinter import ttk, StringVar,filedialog,messagebox,scrolledtext,Button
import copy
import tifffile as tif
import pandas as pd
import subprocess
import skimage
import warnings
from pycromanager import Core
from pycromanager import Acquisition, multi_d_acquisition_events

In [2]:
import sys
import numpy
import pandas
import pycromanager

print(f"Python version: {sys.version}")
print(f"Numpy version: {numpy.__version__}")
print(f"Pandas version: {pandas.__version__}")
print(f"Pycromanager version: {pycromanager.__version__}")

Python version: 3.10.13 | packaged by Anaconda, Inc. | (main, Sep 11 2023, 13:24:38) [MSC v.1916 64 bit (AMD64)]
Numpy version: 1.26.0
Pandas version: 1.5.3
Pycromanager version: 0.29.9


In [3]:
from pycromanager import Core
from pycromanager import Acquisition, multi_d_acquisition_events
import os
import json
import numpy as np
import time
import shutil
import pandas as pd

class scope_constant():
    piezo_focus_start_pos = -30
    piezo_focus_end_pos = 30
    piezo_step = 1.5

    scope_start_pos = -15
    scope_end_pos = 15
    scope_focus_end_pos = 30
    scope_focus_start_pos = -30
    scope_step = 1.5

    piezo_maxpro_start_pos = -15
    piezo_maxpro_end_pos = 15
    sharpen1 = np.array(([0, 1, 0],
                         [-1, 5, -1],
                         [0, -1, 0]), dtype="int")
    pos_per_slice = 4;
pos_path = os.path.join("E:\\","20250404_test_scope")
cycle = "cycle00"
core=Core()

In [4]:
with open(os.path.join("config_file", "scope.json"), 'r') as r:
   scope_cfg=json.load(r)
system_path=os.getcwd()
slicePerSlide =1
focus_status = 0
alignment_status = 0
maketiles_status = 0
max_projection_status = 0
maxprojection_name =''
cancel_process = 0
live_plot=0
focus_bad=0
mock_align=0
ZDrive_safe_pos=scope_cfg[0]["ZDrive_safe_pos"]
XYStage_fluidics_safe_pos=scope_cfg[0]["XYStage_fluidics_safe_pos"]
piezo=scope_cfg[0]["piezo"]
XYStage_image_safe_pos=scope_cfg[0]["XYStage_image_safe_pos"]
scope_exposure_time_dict=scope_cfg[0]["scope_exposure_time_dict"]
stage_x_dir=scope_cfg[0]["stage_x_dir"]
stage_y_dir=scope_cfg[0]["stage_y_dir"]
pixelsize=scope_cfg[0]["pixel_size"]
imwidth=scope_cfg[0]["imwidth"]
overlap=scope_cfg[0]["overlap"]
maxprojection_drive=scope_cfg[0]["maxproject_drive"]
gene_target_channel =scope_cfg[0]["geneseq_focus_target_channel"]
hyb_target_channel=scope_cfg[0]["Hyb_focus_target_channel"]
align_channel=scope_cfg[0]["align_channel"]

In [5]:
scope_exposure_time_dict

{'G': 130,
 'T': 150,
 'A': 70,
 'C': 60,
 'DIC': 10,
 'Hyb-GFP': 100,
 'Hyb-TxRed': 100,
 'Hyb-DAPI': 20,
 'Hyb-Cy5': 100}

In [22]:
core.get_y_position()

1743

In [23]:
core.get_position("DA Z Stage")

200

In [24]:
scope_exposure_time_dict

{'G': 130,
 'T': 150,
 'A': 70,
 'C': 60,
 'DIC': 10,
 'Hyb-GFP': 100,
 'Hyb-TxRed': 100,
 'Hyb-DAPI': 20,
 'Hyb-Cy5': 100}

In [25]:
core.stop_stage_sequence('DA Z Stage')

In [26]:

piezo_event=[{'axes': {'channel': 'DIC', 'z': 0},
  'config_group': ['Channel', 'DIC'],
  'exposure': 20,
  'z': 170.0},
 {'axes': {'channel': 'DIC', 'z': 1},
  'config_group': ['Channel', 'DIC'],
  'exposure': 20,
  'z': 171.5},
 {'axes': {'channel': 'DIC', 'z': 2},
  'config_group': ['Channel', 'DIC'],
  'exposure': 20,
  'z': 173.0},
 {'axes': {'channel': 'DIC', 'z': 3},
  'config_group': ['Channel', 'DIC'],
  'exposure': 20,
  'z': 174.5},
 {'axes': {'channel': 'DIC', 'z': 4},
  'config_group': ['Channel', 'DIC'],
  'exposure': 20,
  'z': 176.0},
 {'axes': {'channel': 'DIC', 'z': 5},
  'config_group': ['Channel', 'DIC'],
  'exposure': 20,
  'z': 177.5},
 {'axes': {'channel': 'DIC', 'z': 6},
  'config_group': ['Channel', 'DIC'],
  'exposure': 20,
  'z': 179.0},
 {'axes': {'channel': 'DIC', 'z': 7},
  'config_group': ['Channel', 'DIC'],
  'exposure': 20,
  'z': 180.5}]

In [27]:
pos_path = os.path.join("E:\\","20250404_test_scope")

In [28]:
core.get_position("DA Z Stage")

200

In [6]:

poslist=pd.read_csv(os.path.join(pos_path,"tiledregoffsetgeneseq01.csv"))
poslist

,Slidenum,Posinfo,X,Y,Z,Pos,switched_Posinfo
0,slide_1,Pos1_000_000,45885,1743,3678.72,Pos1,Pos1_002_005
1,slide_1,Pos1_001_000,46494,1743,3678.72,Pos1,Pos1_001_005
2,slide_1,Pos1_002_000,47102,1743,3678.72,Pos1,Pos1_000_005
3,slide_1,Pos1_000_001,45885,2351,3678.72,Pos1,Pos1_002_004
4,slide_1,Pos1_001_001,46494,2351,3678.72,Pos1,Pos1_001_004
5,slide_1,Pos1_002_001,47102,2351,3678.72,Pos1,Pos1_000_004
6,slide_1,Pos1_000_002,45885,2959,3678.72,Pos1,Pos1_002_003
7,slide_1,Pos1_001_002,46494,2959,3678.72,Pos1,Pos1_001_003
8,slide_1,Pos1_002_002,47102,2959,3678.72,Pos1,Pos1_000_003
9,slide_1,Pos1_000_003,45885,3568,3678.72,Pos1,Pos1_002_002


In [7]:
channel=["A","C","G","T"]
piezo_event_pos = np.arange(185, 215, 1.5)
piezo_event = []
for c in channel:
    for z in range(0, len(piezo_event_pos)):
        piezo_event.append({'axes': {'channel': c, 'z': z},
                            'config_group': ['Channel', c],
                            'exposure': scope_exposure_time_dict.get(c),
                            'z': piezo_event_pos[z]})
       

In [8]:
piezo_event

[{'axes': {'channel': 'A', 'z': 0},
  'config_group': ['Channel', 'A'],
  'exposure': 70,
  'z': 185.0},
 {'axes': {'channel': 'A', 'z': 1},
  'config_group': ['Channel', 'A'],
  'exposure': 70,
  'z': 186.5},
 {'axes': {'channel': 'A', 'z': 2},
  'config_group': ['Channel', 'A'],
  'exposure': 70,
  'z': 188.0},
 {'axes': {'channel': 'A', 'z': 3},
  'config_group': ['Channel', 'A'],
  'exposure': 70,
  'z': 189.5},
 {'axes': {'channel': 'A', 'z': 4},
  'config_group': ['Channel', 'A'],
  'exposure': 70,
  'z': 191.0},
 {'axes': {'channel': 'A', 'z': 5},
  'config_group': ['Channel', 'A'],
  'exposure': 70,
  'z': 192.5},
 {'axes': {'channel': 'A', 'z': 6},
  'config_group': ['Channel', 'A'],
  'exposure': 70,
  'z': 194.0},
 {'axes': {'channel': 'A', 'z': 7},
  'config_group': ['Channel', 'A'],
  'exposure': 70,
  'z': 195.5},
 {'axes': {'channel': 'A', 'z': 8},
  'config_group': ['Channel', 'A'],
  'exposure': 70,
  'z': 197.0},
 {'axes': {'channel': 'A', 'z': 9},
  'config_group': [

## test a simple acuisition

In [14]:
#from pycromanager import Acquisition, multi_d_acquisition_events
#
#with Acquisition( name='acquisition_name',show_display=True) as acq:
#    events = multi_d_acquisition_events(num_time_points=5)
#    acq.acquire(events)

In [35]:
piezo_event

[{'axes': {'channel': 'A', 'z': 0},
  'config_group': ['Channel', 'A'],
  'exposure': 70,
  'z': 170.0},
 {'axes': {'channel': 'A', 'z': 1},
  'config_group': ['Channel', 'A'],
  'exposure': 70,
  'z': 171.5},
 {'axes': {'channel': 'A', 'z': 2},
  'config_group': ['Channel', 'A'],
  'exposure': 70,
  'z': 173.0},
 {'axes': {'channel': 'A', 'z': 3},
  'config_group': ['Channel', 'A'],
  'exposure': 70,
  'z': 174.5},
 {'axes': {'channel': 'A', 'z': 4},
  'config_group': ['Channel', 'A'],
  'exposure': 70,
  'z': 176.0},
 {'axes': {'channel': 'A', 'z': 5},
  'config_group': ['Channel', 'A'],
  'exposure': 70,
  'z': 177.5},
 {'axes': {'channel': 'A', 'z': 6},
  'config_group': ['Channel', 'A'],
  'exposure': 70,
  'z': 179.0},
 {'axes': {'channel': 'A', 'z': 7},
  'config_group': ['Channel', 'A'],
  'exposure': 70,
  'z': 180.5},
 {'axes': {'channel': 'A', 'z': 8},
  'config_group': ['Channel', 'A'],
  'exposure': 70,
  'z': 182.0},
 {'axes': {'channel': 'A', 'z': 9},
  'config_group': [

## test our acquisition code

In [9]:
poslist

,Slidenum,Posinfo,X,Y,Z,Pos,switched_Posinfo
0,slide_1,Pos1_000_000,45885,1743,3678.72,Pos1,Pos1_002_005
1,slide_1,Pos1_001_000,46494,1743,3678.72,Pos1,Pos1_001_005
2,slide_1,Pos1_002_000,47102,1743,3678.72,Pos1,Pos1_000_005
3,slide_1,Pos1_000_001,45885,2351,3678.72,Pos1,Pos1_002_004
4,slide_1,Pos1_001_001,46494,2351,3678.72,Pos1,Pos1_001_004
5,slide_1,Pos1_002_001,47102,2351,3678.72,Pos1,Pos1_000_004
6,slide_1,Pos1_000_002,45885,2959,3678.72,Pos1,Pos1_002_003
7,slide_1,Pos1_001_002,46494,2959,3678.72,Pos1,Pos1_001_003
8,slide_1,Pos1_002_002,47102,2959,3678.72,Pos1,Pos1_000_003
9,slide_1,Pos1_000_003,45885,3568,3678.72,Pos1,Pos1_002_002


In [10]:
pos_path

'E:\\20250404_test_scope'

In [11]:
image_path=os.path.join(pos_path,'geneseq01')
for index, row in poslist.iterrows():
    pos = row['switched_Posinfo']
    x=row['X']
    y=row['Y']
    core.set_xy_position(x, y)
    core.wait_for_device("XYStage")
    print( "XYStage ready")
    z=row['Z']
    core.set_position("ZDrive", z)
    core.wait_for_device("ZDrive")
    print( "ZDrive ready")
    piezo_z=200
    core.set_position("DA Z Stage", piezo_z)
    core.wait_for_device("DA Z Stage")
    print("DA Z Stage")
    with Acquisition(directory=image_path, name=pos,show_display=False) as acq:
        acq.acquire(piezo_event)
        acq.mark_finished()    
    core.stop_stage_sequence('DA Z Stage')
    core.wait_for_device("DA Z Stage")
        

XYStage ready
ZDrive ready
DA Z Stage
XYStage ready
ZDrive ready
DA Z Stage


C:\Users\SVC_barseq\AppData\Local\Temp\ipykernel_16536\3965093562.py:17: ResourceWarning: unclosed file <_io.BufferedReader name='E:\\20250404_test_scope\\geneseq01\\Pos1_002_005_1\\Pos1_002_005_NDTiffStack.tif'>
  with Acquisition(directory=image_path, name=pos,show_display=False) as acq:


XYStage ready
ZDrive ready
DA Z Stage


C:\Users\SVC_barseq\AppData\Local\Temp\ipykernel_16536\3965093562.py:17: ResourceWarning: unclosed file <_io.BufferedReader name='E:\\20250404_test_scope\\geneseq01\\Pos1_001_005_1\\Pos1_001_005_NDTiffStack.tif'>
  with Acquisition(directory=image_path, name=pos,show_display=False) as acq:


XYStage ready
ZDrive ready
DA Z Stage


...

C:\Users\SVC_barseq\AppData\Local\Temp\ipykernel_16536\3965093562.py:17: ResourceWarning: unclosed file <_io.BufferedReader name='E:\\20250404_test_scope\\geneseq01\\Pos1_000_001_1\\Pos1_000_001_NDTiffStack.tif'>
  with Acquisition(directory=image_path, name=pos,show_display=False) as acq:


XYStage ready
ZDrive ready
DA Z Stage


C:\Users\SVC_barseq\AppData\Local\Temp\ipykernel_16536\3965093562.py:17: ResourceWarning: unclosed file <_io.BufferedReader name='E:\\20250404_test_scope\\geneseq01\\Pos1_002_000_1\\Pos1_002_000_NDTiffStack.tif'>
  with Acquisition(directory=image_path, name=pos,show_display=False) as acq:


XYStage ready
ZDrive ready
DA Z Stage


C:\Users\SVC_barseq\AppData\Local\Temp\ipykernel_16536\3965093562.py:17: ResourceWarning: unclosed file <_io.BufferedReader name='E:\\20250404_test_scope\\geneseq01\\Pos1_001_000_1\\Pos1_001_000_NDTiffStack.tif'>
  with Acquisition(directory=image_path, name=pos,show_display=False) as acq:
